# Titanic Survival Heuristic

**Goal**: Build a simple rule‑based model to predict passenger survival on the Titanic, using only the `Sex` feature and age. This will serve as a **baseline** – any future machine learning model must outperform this.

**Dataset**: The classic Titanic training set (891 passengers).

In [2]:
import pandas as pd


In [3]:
print("Pandas version:", pd.__version__)

Pandas version: 2.3.3


In [4]:
# Loading the data
df = pd.read_csv(r'C:\Users\LLC\Downloads\titanic.csv')
# first 5 rows
print(df.head()) 
 # average of the Age column
print("\nAverage age:", df['age'].mean())      

   survived  pclass                                               name  \
0         0       3                            Braund, Mr. Owen Harris   
1         1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2         1       3                             Heikkinen, Miss. Laina   
3         1       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4         0       3                           Allen, Mr. William Henry   

      sex   age     fare  sibsp  parch  
0    male  22.0   7.2500      1      0  
1  female  38.0  71.2833      1      0  
2  female  26.0   7.9250      0      0  
3  female  35.0  53.1000      1      0  
4    male  35.0   8.0500      0      0  

Average age: 29.69911764705882


## Initial Exploration

Let’s understand the data size, column types, and missing values.

In [5]:
print(df.info())
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714 entries, 0 to 713
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  714 non-null    int64  
 1   pclass    714 non-null    int64  
 2   name      714 non-null    object 
 3   sex       714 non-null    object 
 4   age       714 non-null    float64
 5   fare      714 non-null    float64
 6   sibsp     714 non-null    int64  
 7   parch     714 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 44.8+ KB
None
         survived      pclass         age        fare       sibsp       parch
count  714.000000  714.000000  714.000000  714.000000  714.000000  714.000000
mean     0.406162    2.236695   29.699118   34.694514    0.512605    0.431373
std      0.491460    0.838250   14.526497   52.918930    0.929783    0.853289
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.000000
25%      0.000000    1.000000   20.125000    8.0

## Why a Sex‑Based Heuristic?

A crosstab shows that survival strongly depends on sex:
- ~75% of females survived
- ~79% of males died

Therefore, a simple rule “predict survive for female, predict die for male” should achieve decent accuracy.

In [6]:
# Survival proportions per sex
Cross=pd.crosstab(df['sex'],df['survived'], normalize='index')

In [7]:
print(Cross)

survived         0         1
sex                         
female    0.245211  0.754789
male      0.794702  0.205298


## The Heuristic Function

- **If** the passenger is female → predict survived (1)  
- **If** the passenger is male and under 12 → predict survived (1) (children were often prioritised)  
- **Otherwise** → predict did not survive (0)

This function does **not** look at the true `Survived` column – it’s a blind rule.

In [8]:
def predict_survival(row):
    if row['sex'] == 'female':
        return 1
    if row['sex'] == 'male':
        # also treat male children as survivors
        if pd.notna(row['age']) and row['age'] < 12:
            return 1
        else:
            return 0
    return 0

In [9]:
# Apply row‑wise and store in a new column
df['prediction'] = df.apply(predict_survival, axis=1)

# Show a few examples
df[['sex', 'age', 'survived', 'prediction']].head(10)

,sex,age,survived,prediction
0,male,22.0,0,0
1,female,38.0,1,1
2,female,26.0,1,1
3,female,35.0,1,1
4,male,35.0,0,0
5,male,54.0,0,0
6,male,2.0,0,1
7,female,27.0,1,1
8,female,14.0,1,1
9,female,4.0,1,1


## Accuracy of the Heuristic

We compare each prediction to the actual outcome and compute the fraction of correct guesses.

In [10]:
accuracy = (df['prediction'] == df['survived']).mean()
print(f"Baseline Heuristic Accuracy: {accuracy:.4f} ({accuracy:.2%})")

Baseline Heuristic Accuracy: 0.7857 (78.57%)


## Interpretation

The heuristic achieves roughly **78–79% accuracy**, far better than random guessing (50%).  
It is far from perfect because:
- Some women died (e.g., those in lower classes)
- Some men survived (e.g., children, or through other circumstances)

This number will be our **baseline** – any machine learning model (logistic regression, decision trees, etc.)  
must beat this to be considered useful.


## Conclusion

- Simple rules based on domain knowledge can give surprisingly good performance.
- The crosstab analysis directly informed our heuristic.
- We now have a reproducible baseline that we’ll improve upon in later phases.